In [1]:
!wget https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py

--2026-04-13 18:38:25--  https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5913 (5.8K) [text/plain]
Saving to: ‘sascorer.py’

sascorer.py         100%[===================>]   5.77K  --.-KB/s    in 0s      

2026-04-13 18:38:25 (78.7 MB/s) - ‘sascorer.py’ saved [5913/5913]



In [2]:
!wget https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz

--2026-04-13 18:39:27--  https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3848394 (3.7M) [application/octet-stream]
Saving to: ‘fpscores.pkl.gz’

fpscores.pkl.gz     100%[===================>]   3.67M  --.-KB/s    in 0.05s   

2026-04-13 18:39:27 (75.7 MB/s) - ‘fpscores.pkl.gz’ saved [3848394/3848394]



In [3]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 57.9 MB/s eta 0:00:00


In [4]:
import sascorer
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, QED, Lipinski, Fragments
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

In [5]:
# Initialize Filter Catalogs
params_brenk = FilterCatalogParams()
params_brenk.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)
catalog_brenk = FilterCatalog(params_brenk)

params_pains = FilterCatalogParams()
params_pains.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
catalog_pains = FilterCatalog(params_pains)

In [12]:
def process_smiles(input_file, output_file):
    df = pd.read_csv(input_file)
    results = []

    for smiles in df['CANONICAL SMILES']:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            # Calculate Lipinski violations manually
            mw = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            hbd = Lipinski.NumHDonors(mol)
            hba = Lipinski.NumHAcceptors(mol)

            violations = 0
            if mw > 500: violations += 1
            if logp > 5: violations += 1
            if hbd > 5: violations += 1
            if hba > 10: violations += 1

            res = {
                'smiles': smiles,
                'QED': round(QED.qed(mol), 2),
                'SA': round(sascorer.calculateScore(mol), 2),
                'MW': round(mw, 2),
                'LogP': round(logp, 2),
                'Lipinski_Pass': violations <= 1,
                # 'BRENK_Alert': catalog_brenk.HasMatch(mol),
                # 'PAINS_Alert': catalog_pains.HasMatch(mol),
            }
            results.append(res)
        else:
            results.append({'smiles': smiles, 'error': 'Invalid SMILES'})

    output_df = pd.DataFrame(results)
    output_df.to_csv(output_file, index=False)

    display(output_df.head())
    print(f'Results saved to {output_file}')

In [13]:
# To run this, ensure input.csv is uploaded
process_smiles('input.csv', 'output.csv')

,smiles,QED,SA,MW,LogP,Lipinski_Pass
0,[N-]=[N+]=NC1(CO)OC(n2ccc(N)nc2=O)C(F)C1O,0.37,4.58,286.22,-0.95,True
1,NC(=O)NS(=O)(=O)c1ccc(N)cc1,0.58,2.02,215.23,-0.37,True
2,CCCCS(=O)(=O)NC(Cc1ccc(OCCCCC2CCNCC2)cc1)C(=O)O,0.38,2.83,440.61,2.95,True
3,NC(=O)c1cccnc1,0.58,1.69,122.13,0.18,True
4,N#CC(CCN1CCC(C(=O)O)(c2ccccc2)CC1)(c1ccccc1)c1...,0.58,2.48,424.54,5.00,True


Results saved to output.csv


## Ручная проверка

In [ ]:
smiles = "O=c1[nH]c2ncnc2[nH]1"

mol = Chem.MolFromSmiles(smiles)
if mol:
    # Calculate Lipinski violations manually
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)

    violations = 0
    if mw > 500: violations += 1
    if logp > 5: violations += 1
    if hbd > 5: violations += 1
    if hba > 10: violations += 1

    res = {
        'smiles': smiles,
        'QED': round(QED.qed(mol), 2),
        'SA': round(sascorer.calculateScore(mol), 2),
        'MW': round(mw, 2),
        'LogP': round(logp, 2),
        'Lipinski_Pass': violations <= 1,
        # 'BRENK_Alert': catalog_brenk.HasMatch(mol),
        # 'PAINS_Alert': catalog_pains.HasMatch(mol),
    }
    print(res)
else:
    print('Invalid SMILES')

Invalid SMILES


[18:44:13] Can't kekulize mol.  Unkekulized atoms: 3 4 5 6 7
